In [1]:
import os
import platform
import sys

import google.colab

print("Environment: Google Colab")
print("Python:", sys.version)
print("System:", platform.system())
print("Machine:", platform.machine())
print("Logical CPUs:", os.cpu_count())

Environment: Google Colab
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
System: Linux
Machine: x86_64
Logical CPUs: 2


In [2]:
from google.colab import files

uploaded = files.upload()

Saving metrics-colab.zip to metrics-colab.zip


In [3]:
from pathlib import Path
from zipfile import ZipFile

project_dir = Path("/content/metrics-verification")
project_dir.mkdir(parents=True, exist_ok=True)

with ZipFile("metrics-colab.zip") as archive:
    archive.extractall(project_dir)

%cd /content/metrics-verification

for name in [
    "requirements.txt",
    "src/__init__.py",
    "src/metrics.py",
    "tests/test_metrics.py",
    "tests/test_metrics_integration.py",
]:
    print(name, "OK" if Path(name).is_file() else "MISSING")

/content/metrics-verification
requirements.txt OK
src/__init__.py OK
src/metrics.py OK
tests/test_metrics.py OK
tests/test_metrics_integration.py OK


In [4]:
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 571.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.5/386.5 kB 1.7 MB/s eta 0:00:00
  Attempting uninstall: pytest
    Found existing installation: pytest 8.4.2
    Uninstalling pytest-8.4.2:
      Successfully uninstalled pytest-8.4.2
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5


In [1]:
from importlib.metadata import version

print("psutil:", version("psutil"))
print("pytest:", version("pytest"))


psutil: 7.2.2
pytest: 9.1.1


In [3]:
from pathlib import Path

print("Current directory:", Path.cwd())

project_dir = Path("/content/metrics-verification")
print("Project exists:", project_dir.is_dir())
print("Test exists:", (project_dir / "tests/test_metrics.py").is_file())

Current directory: /content
Project exists: True
Test exists: True


In [4]:
%cd /content/metrics-verification

/content/metrics-verification


In [5]:
from pathlib import Path
import subprocess
import sys

Path("tmp").mkdir(exist_ok=True)

result = subprocess.run(
    [
        sys.executable,
        "-m", "pytest",
        "tests/test_metrics.py",
        "tests/test_metrics_integration.py",
        "-q",
        "--basetemp=tmp/pytest-metrics",
        "--tb=short",
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("Exit code:", result.returncode)

........................................................................ [ 26%]
........................................................................ [ 52%]
........................................................................ [ 78%]
..........................................................               [100%]
274 passed in 0.46s

Exit code: 0


In [6]:
from pathlib import Path
from uuid import uuid4
import json

from src.metrics import (
    EventLogger,
    ResourceMonitor,
    build_run_report,
    collect_environment,
    log_resource_results,
    make_request_hook,
    measure_stage,
    read_events,
    write_run_report,
)

run_id = f"colab-demo-{uuid4().hex[:8]}"
run_dir = Path("/content/metrics-verification/experiments") / run_id
work_dir = run_dir / "work"
output_dir = work_dir / "corpus"
output_dir.mkdir(parents=True)

logger = EventLogger(run_dir / "events.jsonl", run_id)
environment = collect_environment()
paper_id = "synthetic-paper"

with ResourceMonitor(work_dir, interval_s=1.0) as monitor:
    with measure_stage(logger, "pipeline"):
        with measure_stage(logger, "discovery"):
            expected_versions = [1]

        with measure_stage(logger, "paper", arxiv_id=paper_id):
            with measure_stage(
                logger, "extract", arxiv_id=paper_id, version=1
            ):
                (output_dir / "main.tex").write_bytes(b"abc")

            with measure_stage(
                logger, "references", arxiv_id=paper_id
            ):
                hook = make_request_hook(
                    logger, "references", arxiv_id=paper_id
                )
                hook(
                    attempt=1,
                    http_status=200,
                    error_code=None,
                    duration=0.01,
                    error_message=None,
                )

paper_results = {
    paper_id: {
        "discovery_status": "ok",
        "expected_versions": expected_versions,
        "versions": {1: {"extract_status": "ok"}},
        "references": {"status": "empty", "references": []},
        "validation_status": "ok",
        "size_measurement_complete": True,
        "unpacked_total_bytes": 10,
        "kept_source_bytes": 3,
    }
}

resources = log_resource_results(logger, monitor.sampler, output_dir)

report = build_run_report(
    run_id=run_id,
    variant="synthetic-colab-demo",
    input_ids=[paper_id],
    paper_results=paper_results,
    events=read_events(logger.path),
    resource_summary=resources,
    environment=environment,
)
write_run_report(run_dir / "metrics.json", report)

print("Output:", run_dir)
print("Completion rate:", report["papers"]["paper_completion_rate"])
print("Output bytes:", report["resources"]["output_storage_bytes"])
print("Observed attempts:", report["requests"]["observed_attempt_count"])

Output: /content/metrics-verification/experiments/colab-demo-87e8bd51
Completion rate: 1.0
Output bytes: 3
Observed attempts: 1


In [7]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

artifact_path = run_dir / "verification-artifacts.zip"

with ZipFile(artifact_path, "w", compression=ZIP_DEFLATED) as archive:
    for name in ["events.jsonl", "metrics.json"]:
        archive.write(run_dir / name, arcname=name)

files.download(str(artifact_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>